### Automated PubChem toxicity data extraction with error handling and data output

This Python script is designed to automate the process of collecting toxicity data, specifically LD50 values, from PubChem for a list of chemical compounds. It begins by reading Compound CIDs and SMILES strings from a CSV file. For each compound, it queries the PubChem PUG View API, using robust error handling and exponential backoff for server issues, to extract relevant 'Non-Human Toxicity Values'. A custom function then recursively parses the JSON response to find and extract all rat oral LD50 values using regular expressions. Finally, it compiles all the collected data, including the CIDs, SMILES, and any extracted LD50 values (or markers for errors/missing data), into a pandas DataFrame and saves the complete and a filtered version (containing only successful LD50 extractions) to separate CSV files.

In [ ]:
# The following Python libraries and modules are required

import pandas as pd
import requests
from urllib.parse import quote
import re
import numpy as np
import time

# Definition of the function that recursively search for all LD50 values in a JSON response using regex

def extract_all_ld50(data):

    ld50_values = []

    ''' The pattern specifically looks for 'LD50 Rat' followed by 'oral' (case-insensitive)
    and then captures the numerical value (including commas and periods) along with any
    preceding comparison operators, ending with 'mg/kg' and optional 'bw'. It captures optional
    comparison symbols (> < ~) before the number '''

    pattern = re.compile(r'LD50 Rat .*?oral.*? ([<>~]?\s*[\d,.]+) mg/kg(?: bw)?', re.IGNORECASE)

    if isinstance(data, dict):
        for key, value in data.items():
            if isinstance(value, (dict, list)):
                # Recursively call function for nested dictionaries or lists
                ld50_values.extend(extract_all_ld50(value))
            elif isinstance(value, str):
                # If the value is a string, search for LD50 pattern
                matches = pattern.findall(value)
                if matches:
                    # Clean extracted values (remove leading/trailing whitespace)
                    # and keep them as strings to preserve operators (e.g., '>1000')
                    ld50_values.extend([m.strip() for m in matches])

    elif isinstance(data, list):
        for item in data:
            # Process lists recursively
            ld50_values.extend(extract_all_ld50(item))

    return ld50_values

# Step 1: Read CID numbers and SMILES from CSV
input_csv = "toxicity.csv"
cols_to_use = ["Compound_CID", "SMILES"]
''' Read Compound_CID as string initially to handle potential non-numeric entries,
then convert with errors='coerce' to turn invalid parsing into NaN, and finally to Int64.
Specifying dtype for SMILES ensures it's treated as a string '''
df = pd.read_csv(input_csv, usecols=cols_to_use, dtype={"Compound_CID": "string", "SMILES": "string"})
df["Compound_CID"] = pd.to_numeric(df["Compound_CID"], errors="coerce").astype("Int64") # Convert to numeric, coerce errors, then to Int64

# Step 2 & 3: Query CIDs, SMILES, LD50s, and build results
all_results = []

max_retries = 5 # Maximum number of retry attempts for API calls
delay = 1 # Initial delay in seconds for exponential backoff

# Define the row range to process for batching or partial execution
start_row = 0
end_row = 11700 # Defines the upper limit (exclusive) for rows to process

# Select the subset of the DataFrame using integer indexing
df_subset = df.iloc[start_row:end_row]

# Iterate through rows of the selected subset DataFrame
for index, row in df_subset.iterrows():
    cid = str(row["Compound_CID"]) # Get Compound_CID for the current row
    smiles = row["SMILES"] # Get SMILES value for the current row

    # Proceed only if CID is not None (i.e., successfully converted to numeric)
    if cid is not None:
        # Construct the PubChem PUG View API URL for toxicity data
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON?heading=Non-Human+Toxicity+Values"
        try:
            r = requests.get(url) # Make the HTTP GET request
            r.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
            response = r.json() # Parse JSON response
            ld50_values = extract_all_ld50(response) # Extract LD50 values using the defined function
            # Store CID, SMILES, and extracted LD50 values in results list
            all_results.append({"CID": cid, "SMILES": smiles, "LD50": ld50_values})
        except requests.exceptions.HTTPError as e: # Catch HTTP-specific errors
            if r.status_code == 404:
                # Handle 404 Not Found: specific heading might not exist for this CID
                print(f"The heading 'Non-Human Toxicity Values' doesn’t exist for CID {cid}: {e}")
                all_results.append({"CID": cid, "SMILES": smiles, "LD50": np.nan}) # Add NaN for missing data
            elif r.status_code == 503:
                # Handle 503 Service Unavailable: implement retry mechanism with exponential backoff
                print(f"[Attempt 1] Server busy (503). Retrying...")
                current_delay = delay # Reset delay for this specific CID's retries
                for attempt in range(max_retries):
                    try:
                        time.sleep(current_delay) # Wait before retrying
                        r = requests.get(url)
                        r.raise_for_status()
                        response = r.json()
                        ld50_values = extract_all_ld50(response)
                        all_results.append({"CID": cid, "SMILES": smiles, "LD50": ld50_values})
                        break # Break out of retry loop if successful
                    except requests.exceptions.HTTPError as e_retry:
                        if r.status_code == 503:
                            # If still 503, increase delay exponentially
                            print(f"[Attempt {attempt+2}] Server busy (503). Retrying in {current_delay}s...")
                            current_delay *= 2 # Exponential backoff
                        else:
                            # If another HTTP error occurs during retry, log and break
                            print(f"Unable to overcome exception 503: {e_retry} (during retry for CID {cid})")
                            all_results.append({"CID": cid, "SMILES": smiles, "LD50": np.nan})
                            break
                    except requests.exceptions.RequestException as e_retry_other:
                        # Catch other request exceptions during retry
                        print(f"Request error during retry for CID {cid}: {e_retry_other}")
                        all_results.append({"CID": cid, "SMILES": smiles, "LD50": np.nan})
                        break
                else:
                    # This block executes if the retry loop finishes without breaking (all retries failed)
                    print(f"Failed to retrieve data for CID {cid} after {max_retries} retries due to 503 errors.")
                    all_results.append({"CID": cid, "SMILES": smiles, "LD50": np.nan}) # Add NaN for errors

            else:
                # Handle other HTTP errors (e.g., 400, 401, 500, etc.)
                print(f"Other HTTP error for CID {cid}: {e}")
                all_results.append({"CID": cid, "SMILES": smiles, "LD50": np.nan}) # Add NaN for errors

        except requests.exceptions.RequestException as e: # Catch all other request-related exceptions
            print(f"Skipping CID {cid} due to general request error: {e}")
            # Include SMILES in the results dictionary even on error
            all_results.append({"CID": cid, "SMILES": smiles, "LD50": np.nan}) # Add NaN for errors
    else:
        # Handle cases where Compound_CID was non-numeric and converted to NaN earlier
        print(f"Skipping row {index}: No valid Compound_CID found (possibly NaN after conversion).")
        # Optionally, you might want to store this row with an indicator that CID was invalid
        all_results.append({"CID": np.nan, "SMILES": smiles, "LD50": np.nan})

# Step 4: Save results to CSV
output_csv = "toxicity_ld50.csv"
df_ld50 = pd.DataFrame(all_results)
df_ld50.to_csv(output_csv, index=False) # Save all results, including those with NaN LD50
print(f"\nFinished. All results (including errors/missing) saved to {output_csv}")

''' Filter the DataFrame to keep rows where 'LD50' is not NaN (meaning an attempt was made and some data was retrieved).
This only filters out explicit np.nan. Since an empty list `[]` for LD50 is not considered NaN, these entries needs to
be removed in a further step '''
df_ld50_filtered_a = df_ld50.dropna(subset=['LD50'])
df_ld50_filtered_b = df_filtered_a[df_filtered_a['LD50'].apply(lambda x: x != '[]')]
output_csv_filtered = "toxicity_ld50_filtered.csv"
df_ld50_filtered_b.to_csv(output_csv_filtered, index=False) # Save only successfully retrieved results
print(f"\nFinished. Filtered results (only successful LD50 extractions) saved to {output_csv_filtered}")